In [1]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261102 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

# COMMAND 

In [2]:
%%writefile association_analysis_prestep.R
#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

# Read input directory, output directory, and chromosome
args <- commandArgs(trailingOnly = TRUE)

# Input parameters
gds_path <- args[1]
output_dir <- args[2]
chr <- as.integer(args[3])

# Print input parameters
cat("CHROMOSOME:", chr, "\n")
cat("INPUT:", gds_path, "\n")
cat("OUTPUT:", output_dir, "\n")

# Annotated GDS filename components
agds_file_name_1 <- "/annotated_chr"
agds_file_name_2 <- ".gds"

###########################################################################
# MAIN FUNCTION
###########################################################################

## load required packages
library(gdsfmt)
library(SeqArray)
library(SeqVarTools)

## channel name of the QC label in the GDS/aGDS file
QC_label <- "annotation/filter"

## annotation name. The first eight names are used to define masks in gene-centric analysis, do not change them!! 
## The others are the annotation you want to use in the STAAR procedure, and they are flexible to change.
name <- c("rs_num","GENCODE.Category","GENCODE.Info","GENCODE.EXONIC.Category",
          "MetaSVM","GeneHancer","CAGE","DHS","CADD","LINSIGHT","FATHMM.XF",
          "aPC.EpigeneticActive","aPC.EpigeneticRepressed","aPC.EpigeneticTranscription",
          "aPC.Conservation","aPC.LocalDiversity","aPC.Mappability","aPC.TF","aPC.Protein")

## channel name of the annotations. Make sure they are matched with the name, especially for the first eight one!! 
dir <- c("/rsid","/genecode_comprehensive_category","/genecode_comprehensive_info",
         "/genecode_comprehensive_exonic_category","/metasvm_pred",
         "/genehancer","/cage_tc","/rdhs","/cadd_phred","/linsight","/fathmm_xf",
         "/apc_epigenetics_active","/apc_epigenetics_repressed","/apc_epigenetics_transcription",
         "/apc_conservation","/apc_local_nucleotide_diversity","/apc_mappability",
         "/apc_transcription_factor","/apc_protein_function")

## Annotation name catalog (alternatively, can skip this part by providing Annotation_name_catalog.csv with the same information)
Annotation_name_catalog <- data.frame(name=name,dir=dir)
# Print output path 
print(paste0(output_dir,"/Annotation_name_catalog_chr",chr,".Rdata",sep=""))
save(Annotation_name_catalog,file=paste0(output_dir,"/Annotation_name_catalog_chr",chr,".Rdata",sep=""))

## Number of jobs for each chromosome
jobs_num <- matrix(rep(0,66),nrow=22)

# Print chromosome processing 
cat("PROCESSING CHROMOSOME:", chr, "\n")

# Open chromosome GDS file
genofile <- seqOpen(gds_path)

# Print genotypes 
print(genofile)

# Read QC labels and identify passing variants
QC_label <- "annotation/info/QC_label"
filter <- seqGetData(genofile, QC_label)

cat("\nQC FIELD:\n")
print(filter)

cat("\nQC CLASS:\n")
print(class(filter))

cat("\nQC VALUES:\n")
print(unique(filter))

cat("\nQC TABLE:\n")
print(table(filter, useNA = "ifany"))

# Read variant positions
position <- as.numeric(seqGetData(genofile, "position"))

cat("\nPOSITIONS:\n")
print(position)

cat("\nPOSITION RANGE:\n")
print(range(position, na.rm = TRUE))

# Read QC labels and identify passing variants
filter <- seqGetData(genofile, QC_label)
SNVlist <- filter == "PASS" 
print(SNVlist)

# Read variant positions
position <- as.numeric(seqGetData(genofile, "position"))

# Record chromosome and position range of PASS variants
jobs_num[chr,1] <- chr
jobs_num[chr,2] <- min(position[SNVlist])
jobs_num[chr,3] <- max(position[SNVlist])

# Close GDS file
seqClose(genofile)

## Individual Analysis
jobs_num <- cbind(jobs_num,ceiling((jobs_num[,3]-jobs_num[,2])/10e6))
## Sliding Window Analysis
jobs_num <- cbind(jobs_num,ceiling((jobs_num[,3]-jobs_num[,2])/5e6))
## Dynamic Window Analysis (SCANG-STAAR)
jobs_num <- cbind(jobs_num,ceiling((jobs_num[,3]-jobs_num[,2])/1.5e6))

# Assign column names 
colnames(jobs_num) <- c("chr","start_loc","end_loc","individual_analysis_num","sliding_window_num","scang_num")
jobs_num <- as.data.frame(jobs_num)

# Print job number
print(jobs_num)

# Print output path 
print(paste0(output_dir,"/jobs_num_chr",chr,".Rdata",sep=""))
save(jobs_num,file=paste0(output_dir,"/jobs_num_chr",chr,".Rdata",sep=""))

Overwriting association_analysis_prestep.R


In [3]:
# Save script to workspace bucket 
!gcloud storage cp association_analysis_prestep.R ${WORKSPACE_BUCKET}/data/scripts/

Copying file://association_analysis_prestep.R to gs://rw-migration-aou-rw-9f879fc4/data/scripts/association_analysis_prestep.R
  Completed files 1/1 | 4.2kiB/4.2kiB                                          


In [4]:
# Check script in workspace bucket 
!gcloud storage cat ${WORKSPACE_BUCKET}/data/scripts/association_analysis_prestep.R 

#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

# Read input directory, output directory, and chromosome
args <- commandArgs(trailingOnly = TRUE)

# Input parameters
gds_path <- args[1]
output_dir <- args[2]
chr <- as.integer(args[3])

# Print input parameters
cat("CHROMOSOME:", chr, "\n")
cat("INPUT:", gds_path, "\n")
cat("OUTPUT:", output_dir, "\n")

# Annotated GDS filename components
agds_file_name_1 <- "/annotated_chr"
agds_file_name_2 <- ".gds"

###########################################################################
# MAIN FUNCTION
###########################################################################

## load required packages
library(gdsfmt)
library(SeqArray)
library(SeqVarTools)

## channel name of the QC label in the GDS/aGDS file
QC_label <- "annotation/filter"

## annotation name. The first eight names are used to define masks in ge

# SUBMIT COMMAND 

In [5]:
import subprocess
import pandas as pd

# Create dataframe of size of exome plink files in GB 
chromosome = []
sizes = [] 

# Loop for all chromosome 
# for chrom in range(1, 22):
for chrom in [2, 4, 15, 17]:
    
    # Extract size of genomic plink bed file in GB for each chromosome 
    command = [
        "gcloud", "storage", "ls",
        "-l",
        f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/annotated_chr{chrom}.gds",
        f"--billing-project={os.environ['GOOGLE_PROJECT']}"
    ]
    
    result = subprocess.run(command, capture_output=True, text=True)

    # Extract genomic size from string 
    lines = result.stdout.splitlines()
    line = lines[0].split()
    
    # Create list genomic size 
    sizes.append(line[0])
    chromosome.append(chrom)
    
# Deposit genomic file size into data frame 
genomic_size = pd.DataFrame({
    "chromosome": chromosome,
    "size": sizes
})

# Format genomic file size 
genomic_size['chromosome'] = genomic_size['chromosome'].astype('int')
genomic_size['size'] = (genomic_size['size'].astype('int') / 1e9) 

print(genomic_size)


   chromosome      size
0          17  0.841535


In [6]:
import shlex

# Logging directory
LOGGING = f"{os.environ['WORKSPACE_BUCKET']}/data/logging"

# Dsub script
SCRIPT = "association_analysis_prestep.R"
PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/scripts/{SCRIPT}"

MAX_DISK_SIZE = 10 # Set maximum disk; adjust for acaf or exome accordingly
EXTRA_DISK_SIZE = 3 # Set additional disk size for each chromosome 

# Loop to submit dsub command
# for CHROM in range(1, 23): 
for CHROM in [17]:
# for CHROM in [17]:

    print(f"Submitting dsub command for chromosome {CHROM}")

    DISK_SIZE = (
        round(
            float(genomic_size.loc[genomic_size["chromosome"] == CHROM, "size"
                ].iloc[0]
            )
        )
        + EXTRA_DISK_SIZE
    )

    print(f"Requesting disk size: {DISK_SIZE}")

    if DISK_SIZE > MAX_DISK_SIZE:
        print(f"Disk too large for chromosome {CHROM}")
        break

    if DISK_SIZE > 1000:
        print(f"Disk too large for chromosome {CHROM}")
        break

    # File paths
    INPUT = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/annotated_chr{CHROM}.gds"
    OUTPUT = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/preparation"

    # R script command
    R_COMMAND = 'Rscript "$SCRIPT" "$INPUT" "$OUTPUT" "$CHROM"'
    
    command = [
        "aou_dsub",
        "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/vieno99/favor:v1.0",
        "--boot-disk-size", "5",
        "--disk-size", str(DISK_SIZE),
        "--machine-type", "n1-standard-1",
        "--logging", LOGGING,
        "--input", f"SCRIPT={PATH}",
        "--input", f"INPUT={INPUT}",
        "--output-recursive", f"OUTPUT={OUTPUT}",
        "--env", f"CHROM={CHROM}",
        "--command", R_COMMAND,
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {shlex.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)

Submitting dsub command for chromosome 17
Requesting disk size: 4
rscript--vienose--261104-012242-03

Job properties:
  job-id: rscript--vienose--261104-012242-03
  job-name: rscript
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/rscript--vienose--261104-012242-03-0-0
Launched job-id: rscript--vienose--261104-012242-03
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261104-012242-03' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261104-012242-03' --users 'vienose'



In [14]:
! dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261104-012242-03' --users 'vienose' --status '*'

Job Name    Status    Last Update
----------  --------  --------------
rscript     SUCCESS   11-04 01:27:26

